## 1. Agglomerative clustering and label assignment

Agglomerative clustering starts with one cluster per attribute and merges clusters according to average pairwise distance. It accepts distances, not similarities, so the blended values are converted with `distance = 1 - similarity`. `n_clusters=None` means the number of groups is not fixed in advance; `distance_threshold = 0.70` controls when merging stops. A lower threshold generally produces more, smaller groups. `labels[i]` is the cluster assigned to `attributes[i]`; the numeric label itself has no semantic ranking.

In [25]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from sklearn.cluster import AgglomerativeClustering
from sklearn.manifold import TSNE

THRESHOLD = 0.70
HIGHLIGHT_GROUPS = 8

with np.load("data/similarity.npz") as data:
    similarity = data["similarity"]
    attributes = data["titles"].tolist()

distance_matrix = 1 - similarity
model = AgglomerativeClustering(
    n_clusters=None,
    metric="precomputed",
    linkage="average",
    distance_threshold=THRESHOLD,
)
labels = model.fit(distance_matrix).labels_

sizes = Counter(labels)
multi = [label for label, size in sizes.items() if size > 1]
print(f"{len(sizes)} clusters: {len(multi)} with multiple attributes, {len(sizes) - len(multi)} singletons")

# Print members so each label can be inspected before the visualization.
members_by_cluster = {
    label: [attributes[index] for index, assigned_label in enumerate(labels) if assigned_label == label]
    for label in sizes
}
for label, members in sorted(members_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])):
    if len(members) > 1:
        print(f"\nCluster {label} ({len(members)} attributes)")
        for member in members:
            print(f"  - {member}")

225 clusters: 102 with multiple attributes, 123 singletons

Cluster 1 (10 attributes)
  - Light insulation
  - Insulating
  - Deep winter insulation
  - Deep winter protection
  - Cool weather insulation
  - Insulating neoprene
  - Winter insulation
  - Highly insulating
  - Thermal insulation
  - Premium deep winter insulation

Cluster 33 (9 attributes)
  - Reflective branding
  - Increased Visibility
  - Reflective details
  - Visibility
  - Reflective detailing
  - Enhanced visibility
  - Reflective detail
  - Stay visible
  - Reflective elements

Cluster 43 (9 attributes)
  - Waterproof and breathable
  - Full waterproof protection
  - Waterproof (8K) and breathable (15K)
  - Waterproof and windproof
  - Waterproof protection
  - Lightweight waterproof protection
  - 100% Waterproof
  - Weather protection 10K (10,000mm)
  - Three-layer waterproof construction

Cluster 23 (8 attributes)
  - Two-way YKK® Vislon zip
  - YKK 2-way zipper
  - Two-way YKK® Vislon AquaGuard zip
  - 2-way 

## Clustering evaluation

We report cluster count, cluster-size distribution, singleton count, coverage, noise, and silhouette using the shared precomputed distance matrix. Agglomerative clustering assigns every attribute to a cluster, so its coverage is 100% and it has no noise points.

DBSCAN's silhouette in `6_dbscan.ipynb` is calculated only on non-noise titles. Its score therefore uses a different subset from the full-data agglomerative score. Compare silhouette together with coverage, cluster sizes, and the actual group members; the silhouette values alone are not directly comparable.

In [26]:
from sklearn.metrics import silhouette_score

cluster_sizes = np.array(list(Counter(labels).values()))
n_points = len(labels)
n_clusters = len(cluster_sizes)
n_noise = int(np.count_nonzero(labels == -1))
n_assigned = n_points - n_noise
coverage = n_assigned / n_points
n_singletons = int(np.count_nonzero(cluster_sizes == 1))

print("Agglomerative clustering evaluation")
print(f"Points assigned: {n_assigned}/{n_points} ({coverage:.1%})")
print(f"Noise points: {n_noise}")
print(f"Clusters: {n_clusters}")
print(f"Singleton clusters: {n_singletons}")
print(f"Cluster sizes (largest first): {sorted(cluster_sizes.tolist(), reverse=True)}")

if 2 <= n_clusters < n_points:
    silhouette = silhouette_score(distance_matrix, labels, metric="precomputed")
    print(f"Silhouette score (all points): {silhouette:.4f}")
else:
    print("Silhouette score not defined (need between 2 and n_points - 1 clusters).")

Agglomerative clustering evaluation
Points assigned: 445/445 (100.0%)
Noise points: 0
Clusters: 225
Singleton clusters: 123
Cluster sizes (largest first): [10, 9, 9, 8, 8, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Silhouette score (all points): 0.1616


Agglomerative clustering evaluation
Points assigned: 445/445 (100.0%)
Noise points: 0
Clusters: 138
Singleton clusters: 53
Cluster sizes (largest first): [20, 15, 15, 14, 14, 13, 12, 12, 10, 10, 9, 8, 7, 7, 7, 7, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Silhouette score (all points): 0.1841
These are results using weights: 0.5, 0.25, 0.25




Top 10 parameter sets meeting minimum coverage (70%), sorted by silhouette desc, noise asc:
eps=0.56, min_samples=2 | silhouette=0.1446, clusters=50, coverage=73.0%, noise=120 (27.0%)
eps=0.58, min_samples=2 | silhouette=0.1117, clusters=46, coverage=76.2%, noise=106 (23.8%)
eps=0.60, min_samples=2 | silhouette=0.0939, clusters=35, coverage=80.9%, noise=85 (19.1%)
eps=0.60, min_samples=3 | silhouette=0.0853, clusters=23, coverage=75.5%, noise=109 (24.5%)
eps=0.62, min_samples=4 | silhouette=0.0692, clusters=12, coverage=72.1%, noise=124 (27.9%)
eps=0.64, min_samples=5 | silhouette=0.0498, clusters=7, coverage=73.3%, noise=119 (26.7%)
eps=0.64, min_samples=4 | silhouette=0.0478, clusters=7, coverage=77.8%, noise=99 (22.2%)
eps=0.68, min_samples=8 | silhouette=0.0474, clusters=2, coverage=73.7%, noise=117 (26.3%)
eps=0.68, min_samples=4 | silhouette=0.0459, clusters=4, coverage=86.3%, noise=61 (13.7%)
eps=0.66, min_samples=5 | silhouette=0.0405, clusters=6, coverage=80.2%, noise=88 (19.8%)

Selected configuration:
eps=0.56, min_samples=2 | silhouette=0.1446, clusters=50, coverage=73.0%, noise=120 (27.0%) 
These results using weights: 0.7 0.15 0.15


Agglomerative clustering evaluation
Points assigned: 445/445 (100.0%)
Noise points: 0
Clusters: 225
Singleton clusters: 123
Cluster sizes (largest first): [10, 9, 9, 8, 8, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Silhouette score (all points): 0.1616
These results using weights: 0.25 0.5 0.25

Agglomerative clustering evaluation
Points assigned: 445/445 (100.0%)
Noise points: 0
Clusters: 238
Singleton clusters: 142
Cluster sizes (largest first): [9, 9, 8, 8, 7, 7, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Silhouette score (all points): 0.1573

giving 75% weights for descriptions and 25 % for titles

## 2. Cluster visualization

t-SNE projects the same precomputed distances to two dimensions for display only; it does not change the clustering labels or serve as an evaluation metric. Nearby points suggest similarity in this projection, but the plot can distort some distances. The largest multi-attribute groups receive distinct colors, other groups are muted, and singleton clusters are shown as hollow markers. Hover over a point to see its attribute and group. The fixed perplexity of 30 requires more than 30 input attributes.

In [27]:
import importlib
import plotly.io._renderers as plotly_renderers

importlib.reload(plotly_renderers)

<module 'plotly.io._renderers' from 'c:\\Users\\greta\\AppData\\Local\\venvs\\baseline-clustering-py313\\Lib\\site-packages\\plotly\\io\\_renderers.py'>

In [28]:
from IPython.display import HTML, display

HIGHLIGHT_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY, INK, MUTED = "#8c8c86", "#c9c8c0", "#1f2933", "#6b7785"

coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    perplexity=30,
    random_state=42,
).fit_transform(distance_matrix)


def group_name(label):
    """Use the most centrally similar attribute as the group's display name."""
    indices = np.where(labels == label)[0]
    within_group_similarity = similarity[np.ix_(indices, indices)]
    medoid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return attributes[medoid_index]


ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]
group_names = {label: group_name(label) for label in multi}


def make_trace(indices, name, color, hollow=False, size=9):
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[attributes[index] for index in indices],
        customdata=[group_names.get(labels[index], "Singleton") for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )


other_group_indices = np.array(
    [index for index, label in enumerate(labels) if sizes[label] > 1 and label not in highlighted_groups]
)
singleton_indices = np.array([index for index, label in enumerate(labels) if sizes[label] == 1])

figure = go.Figure()
figure.add_trace(
    make_trace(singleton_indices, f"Singletons ({len(singleton_indices)})", LIGHT_GRAY, hollow=True, size=7)
)
figure.add_trace(
    make_trace(other_group_indices, f"Other groups ({len(ranked_groups) - len(highlighted_groups)})", GRAY, size=7)
)
for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    figure.add_trace(make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, size=10))

figure.update_layout(
    title=dict(text="Attribute clusters: context-aware similarity (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))